# Weather Data Engineering Pipeline Walkthrough

This notebook demonstrates the end-to-end weather data engineering pipeline:

Open-Meteo API → Python ingestion → PostgreSQL raw layer → dbt staging → dbt mart → Data quality tests → Rerun safety → Business analysis.

The pipeline uses a logical weather date and is designed to be rerun safely without creating duplicate city-date records.

In [1]:
import os
import sys
import subprocess
import pandas as pd
import psycopg2

sys.path.insert(0, "/opt/airflow")

from ingestion.weather import run_pipeline

TARGET_DATE = "2026-09-08"

print("Notebook initialized successfully.")
print("Target date:", TARGET_DATE)

Notebook initialized successfully.
Target date: 2026-09-08


In [2]:
def get_connection():
    return psycopg2.connect(
        host=os.getenv("WAREHOUSE_HOST", "postgres"),
        port=os.getenv("WAREHOUSE_PORT", "5432"),
        dbname=os.getenv("WAREHOUSE_DB", "warehouse"),
        user=os.getenv("WAREHOUSE_USER", "de"),
        password=os.getenv("WAREHOUSE_PASSWORD", "de"),
    )


def query_df(sql):
    with get_connection() as conn:
        return pd.read_sql(sql, conn)


print("Database helper functions are ready.")


Database helper functions are ready.


## 1. Raw Layer

The raw layer stores the weather data extracted from the Open-Meteo archive API.

Each record represents one city for one weather date. The table uses `(city, weather_date)` as its primary key to prevent duplicate city-date records.

In [3]:
raw_summary = query_df("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT city) AS city_count,
    MIN(weather_date) AS earliest_date,
    MAX(weather_date) AS latest_date
FROM raw.weather_daily
""")

display(raw_summary)

/tmp/ipykernel_508/2352458274.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,total_rows,city_count,earliest_date,latest_date
0,96,3,2026-08-10,2026-09-10


In [4]:
raw_sample = query_df("""
SELECT
    city,
    weather_date,
    temperature_2m_max,
    temperature_2m_min,
    temperature_2m_mean,
    precipitation_sum,
    wind_speed_10m_max
FROM raw.weather_daily
ORDER BY weather_date DESC, city
LIMIT 15
""")

display(raw_sample)

/tmp/ipykernel_508/2352458274.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,city,weather_date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,Bengaluru,2026-09-10,30.1,21.0,25.7,1.2,13.7
1,Chennai,2026-09-10,33.7,26.7,29.5,0.4,14.2
2,Mumbai,2026-09-10,29.1,25.1,26.8,2.8,10.2
3,Bengaluru,2026-09-09,30.6,20.8,25.8,0.1,14.6
4,Chennai,2026-09-09,35.0,27.2,30.8,1.0,14.8
5,Mumbai,2026-09-09,29.1,25.7,27.3,3.1,14.7
6,Bengaluru,2026-09-08,31.0,20.4,25.1,6.6,14.9
7,Chennai,2026-09-08,34.8,27.9,30.9,1.0,14.7
8,Mumbai,2026-09-08,29.3,24.8,26.9,6.0,15.3
9,Bengaluru,2026-09-07,30.9,20.8,25.9,0.0,13.3


In [5]:
run_pipeline(TARGET_DATE)

print(f"Weather ingestion completed successfully for {TARGET_DATE}.")

Loaded 3 rows for 2026-09-08
Weather ingestion completed successfully for 2026-09-08.


In [6]:
target_date_raw = query_df(f"""
SELECT
    city,
    weather_date,
    temperature_2m_max,
    temperature_2m_min,
    temperature_2m_mean,
    precipitation_sum,
    wind_speed_10m_max
FROM raw.weather_daily
WHERE weather_date = '{TARGET_DATE}'
ORDER BY city
""")

display(target_date_raw)

assert len(target_date_raw) == 3, "Expected one record for each configured city."

print(f"Raw layer contains {len(target_date_raw)} records for {TARGET_DATE}.")

/tmp/ipykernel_508/2352458274.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,city,weather_date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,Bengaluru,2026-09-08,31.0,20.4,25.1,6.6,14.9
1,Chennai,2026-09-08,34.8,27.9,30.9,1.0,14.7
2,Mumbai,2026-09-08,29.3,24.8,26.9,6.0,15.3


Raw layer contains 3 records for 2026-09-08.


## 2. dbt Transformation Layer

The dbt layer transforms the raw weather data into:

- `public_staging.stg_weather`: cleaned and standardized staging data
- `public_marts.fct_city_daily`: analytical city-level daily weather data

The staging layer prepares the data, while the mart layer provides a business-ready structure for analysis.

In [7]:
dbt_run = subprocess.run(
    ["dbt", "run"],
    cwd="/opt/airflow/dbt",
    capture_output=True,
    text=True
)

print(dbt_run.stdout)

if dbt_run.returncode != 0:
    print(dbt_run.stderr)
    raise RuntimeError("dbt run failed")

print("dbt models completed successfully.")

14:11:13  Running with dbt=1.8.8
14:11:13  Registered adapter: postgres=1.8.2
14:11:14  Found 2 models, 14 data tests, 1 source, 423 macros
14:11:14  
14:11:14  Concurrency: 4 threads (target='dev')
14:11:14  
14:11:14  1 of 2 START sql view model public_staging.stg_weather ......................... [RUN]
14:11:14  1 of 2 OK created sql view model public_staging.stg_weather .................... [CREATE VIEW in 0.21s]
14:11:14  2 of 2 START sql view model public_marts.fct_city_daily ........................ [RUN]
14:11:14  2 of 2 OK created sql view model public_marts.fct_city_daily ................... [CREATE VIEW in 0.09s]
14:11:14  
14:11:14  Finished running 2 view models in 0 hours 0 minutes and 0.54 seconds (0.54s).
14:11:14  
14:11:14  Completed successfully
14:11:14  
14:11:14  Done. PASS=2 WARN=0 ERROR=0 SKIP=0 TOTAL=2

dbt models completed successfully.


In [8]:
staging_data = query_df("""
SELECT
    city,
    weather_date,
    temperature_2m_max,
    temperature_2m_min,
    temperature_2m_mean,
    precipitation_sum,
    wind_speed_10m_max
FROM public_staging.stg_weather
ORDER BY weather_date DESC, city
LIMIT 15
""")

display(staging_data)

/tmp/ipykernel_508/2352458274.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,city,weather_date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,Bengaluru,2026-09-10,30.1,21.0,25.7,1.2,13.7
1,Chennai,2026-09-10,33.7,26.7,29.5,0.4,14.2
2,Mumbai,2026-09-10,29.1,25.1,26.8,2.8,10.2
3,Bengaluru,2026-09-09,30.6,20.8,25.8,0.1,14.6
4,Chennai,2026-09-09,35.0,27.2,30.8,1.0,14.8
5,Mumbai,2026-09-09,29.1,25.7,27.3,3.1,14.7
6,Bengaluru,2026-09-08,31.0,20.4,25.1,6.6,14.9
7,Chennai,2026-09-08,34.8,27.9,30.9,1.0,14.7
8,Mumbai,2026-09-08,29.3,24.8,26.9,6.0,15.3
9,Bengaluru,2026-09-07,30.9,20.8,25.9,0.0,13.3


In [9]:
staging_duplicates = query_df("""
SELECT
    city,
    weather_date,
    COUNT(*) AS duplicate_count
FROM public_staging.stg_weather
GROUP BY city, weather_date
HAVING COUNT(*) > 1
""")

display(staging_duplicates)

assert staging_duplicates.empty, "Duplicate city-date records found in staging."

print("Staging duplicate check passed.")

/tmp/ipykernel_508/2352458274.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,city,weather_date,duplicate_count


Staging duplicate check passed.


In [10]:
mart_data = query_df("""
SELECT
    city,
    weather_date,
    temperature_2m_max,
    temperature_2m_min,
    temperature_2m_mean,
    precipitation_sum,
    wind_speed_10m_max
FROM public_marts.fct_city_daily
ORDER BY weather_date DESC, city
LIMIT 15
""")

display(mart_data)

/tmp/ipykernel_508/2352458274.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,city,weather_date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,Bengaluru,2026-09-10,30.1,21.0,25.7,1.2,13.7
1,Chennai,2026-09-10,33.7,26.7,29.5,0.4,14.2
2,Mumbai,2026-09-10,29.1,25.1,26.8,2.8,10.2
3,Bengaluru,2026-09-09,30.6,20.8,25.8,0.1,14.6
4,Chennai,2026-09-09,35.0,27.2,30.8,1.0,14.8
5,Mumbai,2026-09-09,29.1,25.7,27.3,3.1,14.7
6,Bengaluru,2026-09-08,31.0,20.4,25.1,6.6,14.9
7,Chennai,2026-09-08,34.8,27.9,30.9,1.0,14.7
8,Mumbai,2026-09-08,29.3,24.8,26.9,6.0,15.3
9,Bengaluru,2026-09-07,30.9,20.8,25.9,0.0,13.3


In [11]:
dbt_test = subprocess.run(
    ["dbt", "test"],
    cwd="/opt/airflow/dbt",
    capture_output=True,
    text=True
)

print(dbt_test.stdout)

if dbt_test.returncode != 0:
    print(dbt_test.stderr)
    raise RuntimeError("dbt test failed")

print("All dbt tests completed successfully.")

14:11:18  Running with dbt=1.8.8
14:11:19  Registered adapter: postgres=1.8.2
14:11:19  Found 2 models, 14 data tests, 1 source, 423 macros
14:11:19  
14:11:19  Concurrency: 4 threads (target='dev')
14:11:19  
14:11:19  1 of 14 START test not_null_fct_city_daily_city ................................ [RUN]
14:11:19  2 of 14 START test not_null_fct_city_daily_latitude ............................ [RUN]
14:11:19  3 of 14 START test not_null_fct_city_daily_longitude ........................... [RUN]
14:11:19  4 of 14 START test not_null_fct_city_daily_precipitation_sum ................... [RUN]
14:11:20  2 of 14 PASS not_null_fct_city_daily_latitude .................................. [PASS in 0.19s]
14:11:20  3 of 14 PASS not_null_fct_city_daily_longitude ................................. [PASS in 0.19s]
14:11:20  5 of 14 START test not_null_fct_city_daily_temperature_2m_max .................. [RUN]
14:11:20  6 of 14 START test not_null_fct_city_daily_temperature_2m_mean ................. 

## 3. Rerun Safety and Idempotency

The ingestion process deletes and reloads records for the requested weather date.

Running the same date again should:

- Preserve the number of records
- Preserve the number of cities
- Avoid duplicate `(city, weather_date)` records

In [12]:
before = query_df(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT city) AS city_count
FROM raw.weather_daily
WHERE weather_date = '{TARGET_DATE}'
""")

run_pipeline(TARGET_DATE)

after = query_df(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT city) AS city_count
FROM raw.weather_daily
WHERE weather_date = '{TARGET_DATE}'
""")

duplicates = query_df(f"""
SELECT
    city,
    weather_date,
    COUNT(*) AS duplicate_count
FROM raw.weather_daily
WHERE weather_date = '{TARGET_DATE}'
GROUP BY city, weather_date
HAVING COUNT(*) > 1
""")

print("Before rerun:")
display(before)

print("After rerun:")
display(after)

print("Duplicate records:")
display(duplicates)

assert before.iloc[0]["row_count"] == after.iloc[0]["row_count"]
assert before.iloc[0]["city_count"] == after.iloc[0]["city_count"]
assert duplicates.empty

print("Rerun safety passed: counts unchanged and no duplicates found.")

/tmp/ipykernel_508/2352458274.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


Loaded 3 rows for 2026-09-08
Before rerun:


,row_count,city_count
0,3,3


After rerun:


,row_count,city_count
0,3,3


Duplicate records:


,city,weather_date,duplicate_count


Rerun safety passed: counts unchanged and no duplicates found.


## 4. Business Analysis

This section summarizes weather conditions by city.

The analysis calculates:

- Average temperature
- Highest recorded temperature
- Total precipitation

These metrics can support city-level weather comparisons and operational planning.

In [13]:
business_result = query_df("""
SELECT
    city,
    ROUND(AVG(temperature_2m_mean)::numeric, 2) AS average_temperature,
    ROUND(MAX(temperature_2m_max)::numeric, 2) AS highest_temperature,
    ROUND(SUM(precipitation_sum)::numeric, 2) AS total_precipitation
FROM public_marts.fct_city_daily
GROUP BY city
ORDER BY average_temperature DESC
""")

display(business_result)

/tmp/ipykernel_508/2352458274.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,city,average_temperature,highest_temperature,total_precipitation
0,Chennai,30.42,36.6,79.8
1,Mumbai,27.17,29.7,274.3
2,Bengaluru,24.11,31.0,108.1


## 5. Design Decisions

### Logical date
The pipeline accepts a target weather date instead of relying on the current system date. This makes scheduled execution, backfills, and reruns reproducible.

### Idempotency
The loader deletes existing records for the requested date before inserting the latest records. The primary key `(city, weather_date)` provides an additional uniqueness guarantee.

### Reliability
The API extraction uses request timeouts and retry handling to recover from temporary failures.

### Data quality
dbt tests validate model integrity and data quality. Additional notebook checks verify duplicate city-date records and rerun safety.

### Layered architecture
The raw, staging, and mart layers separate ingestion, transformation, and business analysis responsibilities.

### Orchestration
Airflow schedules the daily pipeline and coordinates ingestion, dbt execution, and dbt testing.